# Transformer Training – Circuit Breaker Health Classification
Binäre Klassifikation: **0 = gesund**, **1 = defekt** (letzte 100 Kurven)

In [15]:
import numpy as np                          # wie MATLAB selbst – Arrays, Mathe, linspace usw.
import matplotlib.pyplot as plt             # wie 'plot()' in MATLAB – zum Zeichnen von Grafiken
import torch                                # PyTorch – die ML-Bibliothek für neuronale Netze
import torch.nn as nn                       # nn = neural network – enthält Layer, Loss-Funktionen usw.
from torch.utils.data import Dataset, DataLoader   # Dataset: Datenbehälter; DataLoader: gibt Batches raus
from sklearn.model_selection import train_test_split  # teilt Daten automatisch in Train/Test auf
from sklearn.metrics import classification_report, confusion_matrix  # Auswertungs-Tools nach dem Training
import seaborn as sns                       # Erweiterung für schönere Plots (wie besseres MATLAB-Figure)

## Schritt 1: Datensatz generieren (mit Rauschen & Verzerrung)

In [ ]:
np.random.seed(42)

n_curves = 1000
t = np.linspace(0, 10, 500)
SEQ_LEN = len(t)

K = 1.0

# D(i) = C - A * exp(k*i), differenzierbar
# Randbedingungen: D(0)=1.5, D(900)=1.0, D(999)≈0
k_opt = 0.01109
A = 0.5 / (np.exp(900 * k_opt) - 1)
C = A + 1.5

i_all    = np.arange(n_curves)
D_values = np.clip(C - A * np.exp(k_opt * i_all), 0.01, None)

D_start     = D_values[0]   # ≈ 1.5
D_threshold = 1.0           # Overshoot-Grenze

# SoH = 1.0 bei i=0 (perfekt gesund), SoH = 0.0 bei i=900 (Defektgrenze), <0 geclippt
soh_values    = np.clip((D_values - D_threshold) / (D_start - D_threshold), 0.0, 1.0).astype(np.float32)
SOH_THRESHOLD = 0.0         # SoH=0 → Defektgrenze

T_values = np.random.uniform(0.5, 2.0, n_curves)

def pt2_step(t, K, T, D):
    """PT2-Sprungantwort analytisch: G(s) = K / (T²s² + 2DTs + 1)"""
    w0 = 1.0 / T
    if D > 1.0:
        wd = w0 * np.sqrt(D**2 - 1)
        y = K * (1 - np.exp(-D*w0*t) * (np.cosh(wd*t) + (D/np.sqrt(D**2-1))*np.sinh(wd*t)))
    elif D == 1.0:
        y = K * (1 - (1 + w0*t) * np.exp(-w0*t))
    else:
        wd = w0 * np.sqrt(1 - D**2)
        y = K * (1 - np.exp(-D*w0*t) * (np.cos(wd*t) + (D/np.sqrt(1-D**2))*np.sin(wd*t)))
    return y

dataset = np.zeros((n_curves, len(t)))

for i in range(n_curves):
    clean      = pt2_step(t, K, T_values[i], D_values[i])
    dataset[i] = clean + np.random.normal(0, 0.02, len(t))

print(f"Dataset Shape: {dataset.shape}")
print(f"SoH: 1.0 bei i=0, 0.0 bei i=900 (Defektgrenze), 0.0 bei i=999 (geclippt)")

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# ── Plot 1: PT2-Kurven ────────────────────────────────────────────────────────
for idx in [0, 450, 899, 900, 950, 999]:
    tag = 'gesund' if soh_values[idx] > SOH_THRESHOLD else 'defekt'
    axes[0].plot(t, dataset[idx], label=f'Kurve {idx} ({tag}, SoH={soh_values[idx]:.2f})', alpha=0.85)
axes[0].set_title('PT2-Kurven mit Rauschen')
axes[0].set_xlabel('Zeit')
axes[0].legend(fontsize=7)
axes[0].grid(True)

# ── Plot 2: Degradationskurve D(i) ───────────────────────────────────────────
axes[1].plot(i_all, D_values, color='steelblue', label='Dämpfungsgrad D(i)')
axes[1].axvline(x=900, color='red',    linestyle='--', label='Defekt-Grenze (i=900)')
axes[1].axhline(y=1.0, color='orange', linestyle=':',  label='D=1 (Overshoot-Grenze)')
axes[1].fill_between(i_all, D_values, 1.0, where=(D_values < 1.0),  alpha=0.15, color='red',   label='Overshoot-Bereich')
axes[1].fill_between(i_all, D_values, 1.0, where=(D_values >= 1.0), alpha=0.1,  color='green', label='Gesund-Bereich')
axes[1].set_title('Degradationskurve')
axes[1].set_xlabel('Kurvenindex i')
axes[1].set_ylabel('Dämpfungsgrad D')
axes[1].legend(fontsize=7)
axes[1].grid(True)

# ── Plot 3: State of Health ───────────────────────────────────────────────────
axes[2].plot(i_all, soh_values, color='seagreen', label='SoH(i)')
axes[2].axvline(x=900, color='red',    linestyle='--', label='Defekt-Grenze (i=900, SoH=0)')
axes[2].axhline(y=0.0, color='orange', linestyle=':',  label='SoH=0 (Defektgrenze)')
axes[2].fill_between(i_all, soh_values, 0, where=(soh_values > 0), alpha=0.1, color='green')
axes[2].set_title('State of Health')
axes[2].set_xlabel('Kurvenindex i')
axes[2].set_ylabel('SoH')
axes[2].set_ylim(-0.05, 1.05)
axes[2].legend(fontsize=7)
axes[2].grid(True)

plt.tight_layout()
plt.show()

## Schritt 2: Dataset & DataLoader

In [17]:
data_min = dataset.min(axis=1, keepdims=True)
data_max = dataset.max(axis=1, keepdims=True)
dataset_norm = (dataset - data_min) / (data_max - data_min + 1e-8)

# Stratifizieren nach gesund/defekt damit beide Klassen im Split gleich verteilt sind
strat_labels = (soh_values < SOH_THRESHOLD).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    dataset_norm, soh_values,
    test_size=0.2,
    random_state=42,
    stratify=strat_labels
)

class CurveDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
        self.y = torch.tensor(y, dtype=torch.float32)   # float für Regression

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(CurveDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(CurveDataset(X_test,  y_test),  batch_size=64, shuffle=False)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Input shape pro Batch: {next(iter(train_loader))[0].shape}")

Train: 800, Test: 200
Input shape pro Batch: torch.Size([32, 500, 1])


## Schritt 3: Transformer Modell

In [18]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class CircuitBreakerTransformer(nn.Module):
    def __init__(self, seq_len=500, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=seq_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=128, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.regressor = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()            # Output in [0, 1] → direkt als SoH interpretierbar
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.transformer(x)
        return self.regressor(x.mean(dim=1)).squeeze(-1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CircuitBreakerTransformer().to(device)
print(model)
print(f"\nParameter: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")

CircuitBreakerTransformer(
  (input_proj): Linear(in_features=1, out_features=64, bias=True)
  (pos_enc): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (regressor): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=

## Schritt 4: Training

In [ ]:
# criterion = nn.MSELoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# def train_epoch(model, loader):
#     model.train()
#     total_loss, correct = 0, 0
#     for X, y in loader:
#         X, y = X.to(device), y.to(device)
#         optimizer.zero_grad()
#         out  = model(X)
#         loss = criterion(out, y)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item() * len(y)
#         # Klassifikations-Accuracy via SoH-Schwelle
#         correct += ((out < SOH_THRESHOLD) == (y < SOH_THRESHOLD)).sum().item()
#     return total_loss / len(loader.dataset), correct / len(loader.dataset)

# @torch.no_grad()
# def eval_epoch(model, loader):
#     model.eval()
#     total_loss, correct = 0, 0
#     for X, y in loader:
#         X, y = X.to(device), y.to(device)
#         out  = model(X)
#         total_loss += criterion(out, y).item() * len(y)
#         correct    += ((out < SOH_THRESHOLD) == (y < SOH_THRESHOLD)).sum().item()
#     return total_loss / len(loader.dataset), correct / len(loader.dataset)

# n_epochs = 30
# history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

# for epoch in range(1, n_epochs + 1):
#     tr_loss, tr_acc = train_epoch(model, train_loader)
#     va_loss, va_acc = eval_epoch(model, test_loader)
#     scheduler.step()
#     history['train_loss'].append(tr_loss)
#     history['train_acc'].append(tr_acc)
#     history['val_loss'].append(va_loss)
#     history['val_acc'].append(va_acc)
#     if epoch % 5 == 0:
#         print(f"Epoch {epoch:02d} | Train MSE: {tr_loss:.5f} Acc: {tr_acc:.3f} | Val MSE: {va_loss:.5f} Acc: {va_acc:.3f}")

KeyboardInterrupt: 

## Modell speichern / laden
**Nach dem Training:** Die "Speichern"-Zelle ausführen.  
**Beim nächsten Start:** Nur Zellen 1–3 (Imports + Klassen) ausführen, dann die "Laden"-Zelle – Training überspringen.

In [ ]:
# # ── Modell SPEICHERN (nach dem Training einmal ausführen) ─────────────────────
# torch.save({
#     'model_state_dict': model.state_dict(),
#     'SEQ_LEN': SEQ_LEN,
# }, 'circuit_breaker_model.pth')

# np.save('dataset_norm.npy', dataset_norm)
# np.save('soh_values.npy',   soh_values)

# print("Modell und Datensatz gespeichert!")

In [ ]:
# ── Modell LADEN (statt Training – Zellen 1-3 vorher ausführen!) ───────────
checkpoint = torch.load('circuit_breaker_model.pth', weights_only=True)
SEQ_LEN = checkpoint['SEQ_LEN']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CircuitBreakerTransformer(seq_len=SEQ_LEN).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

dataset_norm = np.load('dataset_norm.npy')
soh_values   = np.load('soh_values.npy')

print(f"Modell geladen! SEQ_LEN={SEQ_LEN}, Kurven={len(dataset_norm)}, Device={device}")

## Schritt 5: Auswertung

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# ── Loss & Accuracy über Epochen ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train MSE')
axes[0].plot(history['val_loss'],   label='Val MSE')
axes[0].set_title('MSE Loss')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'],   label='Val')
axes[1].set_title('Klassifikations-Accuracy (via SoH-Schwelle)')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

# ── Vorhersagen sammeln ───────────────────────────────────────────────────────
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for X, y in test_loader:
        all_preds.extend(model(X.to(device)).cpu().numpy())
        all_true.extend(y.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

mae = mean_absolute_error(all_true, all_preds)
r2  = r2_score(all_true, all_preds)
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

# ── Scatter: vorhergesagter SoH vs. wahrer SoH ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(all_true, all_preds, alpha=0.6, s=20)
axes[0].plot([0, 1], [0, 1], 'r--', label='Ideal')
axes[0].axvline(SOH_THRESHOLD, color='orange', linestyle=':', label=f'Schwelle {SOH_THRESHOLD:.2f}')
axes[0].axhline(SOH_THRESHOLD, color='orange', linestyle=':')
axes[0].set_xlabel('Wahrer SoH'); axes[0].set_ylabel('Vorhergesagter SoH')
axes[0].set_title(f'SoH Regression  (MAE={mae:.4f}, R²={r2:.3f})')
axes[0].legend(); axes[0].grid(True)

# ── Konfusionsmatrix (via Schwelle) ───────────────────────────────────────────
pred_labels = (all_preds < SOH_THRESHOLD).astype(int)
true_labels = (all_true  < SOH_THRESHOLD).astype(int)
print(classification_report(true_labels, pred_labels, target_names=['gesund', 'defekt']))

cm = confusion_matrix(true_labels, pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['gesund', 'defekt'], yticklabels=['gesund', 'defekt'])
axes[1].set_title('Confusion Matrix (SoH-Schwelle)')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.tight_layout(); plt.show()

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt

# ── Modell-Klassen ──────────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class CircuitBreakerTransformer(nn.Module):
    def __init__(self, seq_len=500, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        self.pos_enc    = PositionalEncoding(d_model, max_len=seq_len)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=128, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.regressor   = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1), nn.Sigmoid())
    def forward(self, x):
        x = self.input_proj(x); x = self.pos_enc(x); x = self.transformer(x)
        return self.regressor(x.mean(dim=1)).squeeze(-1)

SOH_THRESHOLD = 1.0 / 1.5   # ≈ 0.667

# ── Index auswählen ─────────────────────────────────────────────────────────
idx = 999   # ← hier ändern (0–999)

# ── Modell & Daten laden ────────────────────────────────────────────────────
checkpoint   = torch.load('circuit_breaker_model.pth', weights_only=True)
SEQ_LEN      = checkpoint['SEQ_LEN']
device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model        = CircuitBreakerTransformer(seq_len=SEQ_LEN).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

dataset_norm = np.load('dataset_norm.npy')
soh_values   = np.load('soh_values.npy')

# ── Vorhersage ──────────────────────────────────────────────────────────────
kurve = torch.tensor(dataset_norm[idx].reshape(1, SEQ_LEN, 1), dtype=torch.float32).to(device)
with torch.no_grad():
    soh_pred = model(kurve).item()

true_soh  = soh_values[idx]
true_tag  = 'gesund' if true_soh  >= SOH_THRESHOLD else 'defekt'
pred_tag  = 'gesund' if soh_pred  >= SOH_THRESHOLD else 'defekt'

plt.figure(figsize=(10, 4))
plt.plot(dataset_norm[idx])
plt.title(f"Kurve {idx} | Wahrheit: {true_tag} (SoH={true_soh:.3f}) | "
          f"Vorhersage: {pred_tag} (SoH={soh_pred:.3f})")
plt.xlabel("Zeitschritte"); plt.ylabel("Amplitude (normiert)")
plt.grid(True); plt.show()